# M0 Pretrained Baseline Training
    ## Objective and hypothesis
    Establish a minimal-augmentation pretrained baseline.
    Fine-tune ImageNet-pretrained MobileNetV3-Small with seven logits on the same audited manifests.
    The controlled variable is training augmentation; this experiment makes no claim to infer internal mood.
    All metrics below must come from this notebook's execution, and the held-out test is untouched here.

## Verify this notebook's GPU kernel
Select the project `.venv`; this is the first executable cell and forbids CPU fallback.

In [ ]:
import sys
import torch
import torchvision
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU-ONLY TRAINING: CUDA unavailable in this kernel. Fix NVIDIA driver, CUDA wheel, interpreter or notebook kernel. CPU training is forbidden.")
device = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU properties:", torch.cuda.get_device_properties(0))
x = torch.randn((128, 128), device=device)
y = x @ x
assert y.is_cuda and y.device.type == "cuda"
torch.cuda.synchronize()
print("GPU tensor test PASSED:", y.device)
del x, y

## Imports, seed and visible configuration
AdamW, cosine schedule, full-network fine-tuning, 30-epoch limit and patience 8. Batch size starts at 32. OOM requires an explicitly recorded change, not automatic CPU fallback.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open the FaceAttend project folder.")
sys.path.insert(0, str(ROOT / "src"))
os.environ["TORCH_HOME"] = str(ROOT / ".cache/torch")
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from faceattend.utils import environment, load_config, seed_everything, write_json

In [ ]:
config = load_config(ROOT, "m0_pretrained_baseline")
RESUME = False  # Set True only to restore this run's last completed epoch.
SHOW_PRIVATE_SAMPLES = False
# Edit documented settings here BEFORE starting a new run, e.g. config["batch_size"] = 16.
if not config["authorized_data"] or not config["label_mapping_verified"]:
    raise RuntimeError("Complete authorized-use and numeric label verification in common.yaml first.")
seed_everything(config["seed"])
output = ROOT / "outputs" / config["experiment"]
output.mkdir(parents=True, exist_ok=True)
if (output / "completed.json").exists():
    raise RuntimeError("This run is already complete. Use comparison notebook; preserve frozen experiments.")
if (output / "last.pt").exists() and not RESUME:
    raise RuntimeError("A run already exists. Set RESUME=True instead of overwriting it.")
config["environment"] = environment()
display(pd.Series(config))

## Dataset and augmentation methodology
RGB direct resize to 224×224 and ImageNet normalization are identical for validation and inference. M0 has no random transformations; M1 adds ±10° rotation, 3% translation and modest brightness/contrast jitter. No vertical flip, rebalancing or aggressive crop.

In [ ]:
from faceattend.data import make_loaders, load_manifests, make_transform
from faceattend import MEAN, STD
tables, fingerprints = load_manifests(ROOT, config)
config["manifest_fingerprints"] = fingerprints
loaders = make_loaders(ROOT, config)
display(pd.DataFrame({name: table.class_name.value_counts() for name, table in tables.items() if name != "test"}).fillna(0))
print("Training transform:", make_transform(config, training=True))
print("Validation transform:", make_transform(config, training=False))

## Optional local augmentation preview
Use a training image only. Clear face-image outputs before sharing or committing.

In [ ]:
if SHOW_PRIVATE_SAMPLES:
    from PIL import Image
    from torchvision.transforms.functional import to_pil_image
    image = Image.open(ROOT / tables["train"].iloc[0].path).convert("RGB")
    augmentation = make_transform(config, training=True)
    fig, axes = plt.subplots(1, 4, figsize=(10, 3))
    for axis in axes:
        tensor = augmentation(image)
        visible = tensor * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
        axis.imshow(to_pil_image(visible.clamp(0, 1)))
        axis.axis("off")
    fig.suptitle("Training image transformations — private local preview")
    plt.show()
# Reset RNG after previews to preserve experimental control.
seed_everything(config["seed"])

## Pretrained model, optimizer, scheduler and AMP
Both runs load the exact IMAGENET1K_V1 weights and fine-tune all layers. Missing pretrained weights must stop training rather than substitute random initialization.

In [ ]:
from faceattend.models import build_model
from faceattend.engine import train_one_epoch, validate_one_epoch, assert_gpu, save_checkpoint, resume_checkpoint
from faceattend.metrics import plot_history, save_metrics
from faceattend.utils import file_hash
model = build_model(pretrained=True).to(device)
assert_gpu(model)
criterion = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])
scaler = torch.amp.GradScaler("cuda", enabled=config["amp"], init_scale=config["amp_initial_scale"])
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("AMP enabled:", scaler.is_enabled(), "Device:", next(model.parameters()).device)
history, best_key, stale, start_epoch = [], (-1., -1., float("-inf")), 0, 1
if RESUME:
    saved = resume_checkpoint(output / "last.pt", model, config, optimizer, scheduler, scaler, loaders)
    history, best_key, stale = saved["history"], tuple(saved["best_key"]), saved["stale"]
    start_epoch = saved["epoch"] + 1
    pd.DataFrame(history).to_csv(output / "history.csv", index=False)
write_json(output / "config_resolved.json", config)

## Visible training orchestration
    Select checkpoints by validation macro F1, then accuracy, then loss. Early stopping resets only on a strict F1 improvement.
    Save resumable `last.pt` and selected `best.pt` after epochs; disk logs survive cleared notebook output.
    Restart after interruption by rerunning from the top with `RESUME=True`. An interrupted epoch is repeated from the previous saved RNG state.
    On OOM, inspect usage and record a smaller batch size for a fresh, comparable pair of runs.

In [ ]:
import time
from tqdm.auto import tqdm
run_start = time.perf_counter()
try:
    for epoch in tqdm(range(start_epoch, config["epochs"] + 1), desc=config["experiment"]):
        if stale >= config["patience"]:
            break
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        epoch_start = time.perf_counter()
        learning_rate = optimizer.param_groups[0]["lr"]
        train = train_one_epoch(model, loaders["train"], criterion, optimizer, scaler, device, config["amp"])
        val = validate_one_epoch(model, loaders["val"], criterion, device, config["amp"])
        torch.cuda.synchronize()
        row = {"epoch": epoch, "train_loss": train["loss"], "val_loss": val["loss"],
               "train_acc": train["accuracy"], "val_acc": val["accuracy"], "val_macro_f1": val["macro_f1"],
               "learning_rate": learning_rate, "epoch_seconds": time.perf_counter() - epoch_start,
               "peak_gpu_memory_mb": torch.cuda.max_memory_allocated() / 2**20,
               "optimizer_steps": train["optimizer_steps"], "amp_skipped_steps": train["amp_skipped_steps"]}
        history.append(row)
        key = (val["macro_f1"], val["accuracy"], -val["loss"])
        stale = 0 if key[0] > best_key[0] else stale + 1
        improved = key > best_key
        if improved:
            best_key = key
        scheduler.step()
        if improved:
            save_checkpoint(output / "best.pt", model, config, epoch, val, history, best_key, stale, optimizer, scheduler, scaler, loaders)
        save_checkpoint(output / "last.pt", model, config, epoch, val, history, best_key, stale, optimizer, scheduler, scaler, loaders)
        pd.DataFrame(history).to_csv(output / "history.csv", index=False)
        print(f"Epoch {epoch}: train loss={train['loss']:.4f}; val accuracy={val['accuracy']:.4f}; macro F1={val['macro_f1']:.4f}; GPU MB={row['peak_gpu_memory_mb']:.1f}")
except torch.cuda.OutOfMemoryError:
    print("CUDA OOM: training stopped. Record a lower batch size and restart a controlled pair of runs; no CPU fallback.")
    raise
except KeyboardInterrupt:
    print("Interrupted: resume last.pt from the previous completed epoch. Do not treat this run as finished.")
    raise
write_json(output / "completed.json", {"last_epoch": history[-1]["epoch"], "stop_reason": "patience" if stale >= config["patience"] else "epoch_limit", "session_seconds": time.perf_counter() - run_start})

## Authentic validation results
Reload the selected checkpoint and report its recorded metrics. Held-out testing is reserved for notebook 04.

In [ ]:
selected = torch.load(output / "best.pt", map_location="cpu", weights_only=True)
display(pd.DataFrame(history).tail())
plot_history(history, output / "training_curves.png")
plt.show()
save_metrics(selected["validation_metrics"], output, "validation")
write_json(output / "metrics.json", selected["validation_metrics"])
plt.show()
print("Best epoch:", selected["epoch"], "Validation macro F1:", selected["validation_macro_f1"])

## Interpretation and limitations
    Results exist only after successful execution. A single seed does not quantify run-to-run uncertainty.
    Class imbalance motivates macro F1 and per-class inspection. GPU nondeterminism may still vary across hardware/library versions.
    Augmentation may help or hurt; do not select settings based on the held-out test. Predictions remain visual categories with uncalibrated scores.